# 3 · Validation and the DiscreteProblem IR

Two things the formalism gives you *before* and *around* the numerics:

1. a **structural validator** that catches malformed models with precise
   diagnostics — no solve required;
2. an inspectable **intermediate representation** (the `DiscreteProblem`) that
   makes the FEniCSx assembly decisions explicit and testable.

And models outside the backend's supported subset **fail loudly** at assembly.

In [1]:
from vcell_fenics.formalism import load_yaml, validate
from vcell_fenics.backend import assemble, make_disk_membrane_geometry

## The validator catches structural errors

This model plants three mistakes in one `source` expression: a second-order
operator on a P1 variable, calculus applied to the equation's *own* variable
(the "narrow rule"), and an undefined name. Each is reported with its location.

In [2]:
BAD = """
math_description:
  geometry: disk_membrane
  subdomains: [{ name: membrane, kind: surface }]
  variables: [{ name: rho, subdomain: membrane }]
  equations:
    - template: surface_pde_with_dilution
      variable: rho
      subdomain: membrane
      temporality: time_dependent
      terms: { diffusion: "0.1", source: "lapl_beltrami(rho) + k_missing" }
      initial_condition: "1.0"
"""
for d in validate(load_yaml(BAD)):
    print(f"[{d.severity}] {d.path}: {d.message}")

[error] equations[0].terms.source: lapl_beltrami(...) may not be applied to the equation's own variable 'rho' in a template slot (§1.11.9, narrow rule); calculus on other variables is allowed
[error] equations[0].terms.source: lapl_beltrami(rho) requires 'rho' to use 'lagrange_p2' or higher; its space is 'lagrange_p1', and the strong second derivative of a P1 field is degenerate (§1.11.9). Raise the variable's space or use the weak-form escape hatch.
[error] equations[0].terms.source: unknown name 'k_missing'


## The IR makes assembly decisions inspectable

A static membrane assembles to `{TIME_DERIVATIVE, DIFFUSION}`; the same model on
a *moving* membrane gains a `DILUTION` term — assertable without solving.

In [3]:
def model(motion: str) -> str:
    return f"""
math_description:
  geometry: disk_membrane
  subdomains: [{{ name: membrane, kind: surface, motion: {motion} }}]
  variables: [{{ name: rho, subdomain: membrane }}]
  equations:
    - template: surface_pde_with_dilution
      variable: rho
      subdomain: membrane
      temporality: time_dependent
      terms: {{ diffusion: "0.1" }}
      initial_condition: "1.0"
  parameters: [{{ name: r_dot, value: 1.0 }}]
"""

geometry = make_disk_membrane_geometry("disk_membrane", surface_subdomain="membrane", h=0.2)
static = assemble(load_yaml(model("{ kind: none }")), geometry, dt=0.01)
moving = assemble(load_yaml(model("{ kind: prescribed, velocity: \"r_dot * x / r(x)\" }")), geometry, dt=0.01)
print("static membrane:", sorted(t.name for t in static.term_kinds()))
print("moving membrane:", sorted(t.name for t in moving.term_kinds()))

static membrane: ['DIFFUSION', 'TIME_DERIVATIVE']
moving membrane: ['DIFFUSION', 'DILUTION', 'TIME_DERIVATIVE']


## Out-of-subset models fail loudly

The v1 backend implements a strict subset. Anything outside it raises a clear
`NotImplementedError` at assembly rather than silently producing a wrong answer.

In [4]:
WEAK_FORM = """
math_description:
  geometry: disk_membrane
  subdomains: [{ name: membrane, kind: surface, motion: { kind: unknown, variable: v } }]
  variables: [{ name: v, subdomain: membrane, type: vector }]
  equations:
    - template: weak_form
      variable: v
      subdomain: membrane
      temporality: steady_state
      form: "inner(v, v_test) * dx_Gamma"
      initial_condition: "0"
"""
try:
    assemble(load_yaml(WEAK_FORM), make_disk_membrane_geometry("disk_membrane", surface_subdomain="membrane", h=0.3), dt=0.01)
except NotImplementedError as e:
    print("rejected:", e)

rejected: backend v1 supports templates ['bulk_radv_diff', 'surface_pde_with_dilution'], not 'weak_form'


**Takeaway.** Correctness is layered: structural validation, then an inspectable
IR (term tags, function spaces), then operator-invariant and analytical checks,
then the solve. A model is either supported and correct, or rejected with a
clear reason — never silently wrong.